# PWC-Net — cross-validation results

Compares, per in-vivo fold, the three PWC-Net variants against **MITT**:

| variant | trained on | colour |
|---|---|---|
| `zero_shot` | simulated fold *k* only | blue |
| `unsup` | + in-vivo train patients, photometric + smoothness loss | orange |
| `mitt` | + in-vivo train patients, MITT contour point loss | aqua |

MITT is drawn as a **white dashed** contour. It is a speckle tracker, not ground truth, so *distance to MITT* is a disagreement, not an error. Only the simulated section has true errors.

Run the pipeline first (`python -m motion_estimator.methods.pwcnet.run_cv`), then set `RUN_DIR` below.

In [ ]:
import json, os, sys
from pathlib import Path

os.environ.setdefault("MKL_THREADING_LAYER", "TBB")
ROOT = Path.cwd()
while not (ROOT / "motion_estimator").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from motion_estimator.methods.pwcnet import plots
from motion_estimator.methods.pwcnet.evaluate import load_rows, summary_table, IN_VIVO_METRICS, SIM_METRICS
from motion_estimator.methods.pwcnet.data import load_folds, load_invivo_clip

from motion_estimator.runs import DEFAULT_RUNS_ROOT as RUNS
# None -> newest pwcnet-cv run; or paste a path
RUN_DIR = None
if RUN_DIR is None:
    RUN_DIR = sorted(p for p in RUNS.iterdir() if "pwcnet-cv" in p.name)[-1]
RUN_DIR = Path(RUN_DIR)
EVAL = RUN_DIR / "eval"
folds = load_folds(RUN_DIR)
print(RUN_DIR)

## 1. Folds — patients per split

In [ ]:
pd.DataFrame([
    {"cohort": cohort, "fold": f["fold"], "train": len(f["train"]), "val": len(f["val"]),
     "test": len(f["test"]), "test patients": ", ".join(p.split("/")[-1] for p in f["test"])}
    for cohort in ("simulated", "in_vivo") for f in folds[cohort]])

## 2. Training curves
Validation end-point error per stage and fold (for `unsup`, `best.pt` is chosen on the photometric loss; the EPE shown is vs MITT on validation patients, for reference only).

In [ ]:
stages = {"sim_fold{}": "simulated pretraining (EPE vs scatterers)",
          "invivo_fold{}_unsup": "in vivo unsupervised (EPE vs MITT)",
          "invivo_fold{}_mitt": "in vivo MITT-supervised (EPE vs MITT)"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (pattern, title) in zip(axes, stages.items()):
    for k in range(folds["k"]):
        path = RUN_DIR / pattern.format(k) / "history.json"
        if path.is_file():
            h = json.loads(path.read_text())["history"]
            ax.plot([r["step"] for r in h], [r["val_epe"] for r in h], label=f"fold {k}", linewidth=2)
    ax.set_title(title, fontsize=10); ax.set_xlabel("step"); ax.set_ylabel("val EPE (px)")
    ax.grid(alpha=0.25); ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

## 3. In vivo — summary across folds
Mean (std) over folds of each fold's median. Read `error_px_es` and `coverage_es` first: end-systole is the frame strain is read at.

In [ ]:
rows_iv = load_rows(RUN_DIR, "in_vivo")
iv = pd.DataFrame(rows_iv)
if "gls_pred" in iv:
    iv["gls_abs_diff_mitt"] = (iv.gls_pred - iv.gls_mitt).abs()
    iv["gls_abs_diff_ge"] = (iv.gls_pred - iv.gls_ge).abs()
summary_table(rows_iv, IN_VIVO_METRICS).round(2)

In [ ]:
# per fold medians, so you can see whether one fold drives the mean
iv.groupby(["fold", "group"])[["error_px_es", "error_px_median", "gls_abs_diff_mitt", "gls_abs_diff_ge"]].median().round(2).unstack("group")

In [ ]:
plots.plot_fold_box(rows_iv, "error_px_es", "ES distance to MITT (px)")
plots.plot_fold_box(rows_iv, "error_px_median", "median distance to MITT over the cycle (px)")
plots.plot_gls_agreement(rows_iv, "gls_mitt")
plots.plot_gls_agreement(rows_iv, "gls_ge");

In [ ]:
# paired comparison: does fine-tuning help on the same clip?
wide = iv.pivot_table(index=["fold", "key"], columns="group", values="error_px_es")
cols = list(wide.columns)
if len(cols) > 1:
    base = [c for c in cols if c.startswith("zero_shot")][0]
    for c in cols:
        if c == base: continue
        d = (wide[c] - wide[base]).dropna()
        print(f"{c:26s} vs {base}: median change {d.median():+.2f} px, better on {100*(d<0).mean():.0f}% of {len(d)} clips")
wide.round(2).head(20)

## 4. Simulated — true errors on held-out patients

In [ ]:
rows_sim = load_rows(RUN_DIR, "simulated")
if rows_sim:
    display(summary_table(rows_sim, SIM_METRICS).round(3))
    plots.plot_fold_box(rows_sim, "error_px_es", "ES error vs scatterers (px)")
else:
    print("no simulated evaluation yet")

## 5. One clip, all variants over the images
Pick a fold and a clip. Saved tracks are loaded from `eval/fold<k>/tracks/`; nothing is recomputed.

In [ ]:
FOLD = 0
PAIRING = "consecutive"
track_files = sorted((EVAL / f"fold{FOLD}" / "tracks").glob("*.npz"))
print(len(track_files), "clips:", ", ".join(p.stem for p in track_files))
CLIP = track_files[0].stem if track_files else None   # e.g. "Retest_dataset_IV02_4ch" 

In [ ]:
def load_tracks(fold, name, pairing=PAIRING):
    data = dict(np.load(EVAL / f"fold{fold}" / "tracks" / f"{name}.npz", allow_pickle=False))
    clip = load_invivo_clip(str(data["cache_path"]))
    n = data["reference_x"].shape[0]
    frames = clip.frames[:n]
    tracks, strains = {}, {}
    for v in plots.VARIANTS:
        if f"x_{v}_{pairing}" in data:
            tracks[v] = (data[f"x_{v}_{pairing}"], data[f"y_{v}_{pairing}"])
            strains[v] = data[f"strain_{v}_{pairing}"]
    return data, frames, tracks, strains

data, frames, tracks, strains = load_tracks(FOLD, CLIP)
plots.plot_clip(frames, data["reference_x"], data["reference_y"], tracks,
                int(data["start_frame"]), int(data["es_frame"]), strains,
                data["mitt_strain"], float(data["gls_ge"]), title=f"{data['key']} -- fold {FOLD}");

### Contours across the cycle
Eight frames from start to end, MITT dashed white, each variant in its colour.

In [ ]:
def cycle_strip(frames, data, tracks, n=8):
    idx = np.linspace(0, frames.shape[0] - 1, n).astype(int)
    es = int(data["es_frame"]) - 1
    if es not in idx:
        idx[np.argmin(abs(idx - es))] = es
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3))
    for ax, i in zip(axes, idx):
        plots.plot_contours(ax, frames[i], data["reference_x"], data["reference_y"], tracks, i,
                            f"frame {i+1}" + (" (ES)" if i == es else ""))
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(l), frameon=True, facecolor="0.3", labelcolor="white", fontsize=9)
    fig.tight_layout(rect=(0, 0.1, 1, 1))

cycle_strip(frames, data, tracks)

### Scroll through frames (needs `ipywidgets`; falls back to the strip above)

In [ ]:
try:
    from ipywidgets import interact, IntSlider
    def show(frame=int(data["es_frame"]) - 1):
        fig, ax = plt.subplots(figsize=(6, 6))
        plots.plot_contours(ax, frames[frame], data["reference_x"], data["reference_y"], tracks, frame, f"frame {frame+1}")
        ax.legend(loc="lower right", facecolor="0.3", labelcolor="white", fontsize=8)
        plt.show()
    interact(show, frame=IntSlider(min=0, max=frames.shape[0]-1, value=int(data["es_frame"])-1))
except ImportError:
    print("pip install ipywidgets for the slider")

## 6. Flow fields — what each model actually predicts
Dense flow for one frame pair (start → ES) of the selected clip, per variant. Magnitude in px, arrows subsampled, sector only.

In [ ]:
import torch
from motion_estimator.methods.pwcnet.estimator import PWCNetEstimator, load_model
from motion_estimator.methods.pwcnet.evaluate import checkpoints

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip = load_invivo_clip(str(data["cache_path"]))
source, target = int(data["start_frame"]), int(data["start_frame"]) + 1   # consecutive pair
ckpts = checkpoints(RUN_DIR, FOLD)
fig, axes = plt.subplots(1, len(ckpts) + 1, figsize=(5 * (len(ckpts) + 1), 5))
axes[0].imshow(clip.frames[source], cmap="gray"); axes[0].set_title(f"frame {source+1}"); axes[0].axis("off")
fields = {}
for ax, (name, path) in zip(axes[1:], ckpts.items()):
    est = PWCNetEstimator(sector=clip.sector, model=load_model(path, device), device=device)
    flow = est.flow(clip.frames[source], clip.frames[target], (0.0, 255.0))
    fields[name] = flow
    mag = np.where(clip.sector, np.hypot(flow[..., 0], flow[..., 1]), np.nan)
    im = ax.imshow(mag, cmap="Blues", vmin=0, vmax=np.nanpercentile(mag, 99))
    s = 16
    yy, xx = np.mgrid[0:mag.shape[0]:s, 0:mag.shape[1]:s]
    keep = clip.sector[::s, ::s]
    ax.quiver(xx[keep], yy[keep], flow[::s, ::s, 0][keep], -flow[::s, ::s, 1][keep], color="0.15", scale_units="xy", angles="xy", scale=0.2, width=0.003)
    ax.plot(data["reference_x"][source] - 1, data["reference_y"][source] - 1, **plots.REFERENCE)
    ax.set_title(plots.LABELS[name], fontsize=10); ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, label="|flow| (px)")
fig.tight_layout()

## 7. Best and worst clips per variant

In [ ]:
for group, sub in iv.groupby("group"):
    s = sub.sort_values("error_px_es")[["fold", "key", "error_px_es", "error_px_median", "gls_pred", "gls_mitt", "gls_ge"]]
    print(f"--- {group}: best 3 / worst 3 by ES distance to MITT")
    display(pd.concat([s.head(3), s.tail(3)]).round(2))

## 8. Saved plots gallery
The PNGs `evaluate.py` wrote for every held-out clip, per fold.

In [ ]:
GALLERY_FOLD = 0
LIMIT = 6
for png in sorted((EVAL / f"fold{GALLERY_FOLD}" / "plots").glob(f"*_{PAIRING}.png"))[:LIMIT]:
    print(png.name)
    display(Image(filename=str(png)))

In [ ]:
for png in ("box_in_vivo_es.png", "gls_vs_mitt.png", "gls_vs_ge.png", "box_simulated_es.png"):
    if (EVAL / png).is_file():
        display(Image(filename=str(EVAL / png)))